In [6]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, plot_tree  # <-- import plot_tree here
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# ──────────────────────────────────────────────────────────
# 1. LOAD  +  CHOOSE COLUMNS
# ──────────────────────────────────────────────────────────
df = pd.read_csv("/Users/derek/Downloads/final_BES.csv")

# -- TARGET (BES) AND FEATURE COLUMNS ------------------------------------------
BES_COL  = 'Volume_Adjusted_Skill_Player_Downfield_BES'   # overall score
FEAT_1   = 'median_ygpb'                                   # post-block yards
FEAT_2   = 'normalized_block_volume'                       # workload (0-1)
FEAT_3   = 'median_Skill_Player_Downfield_BES'             # quality signal
# ------------------------------------------------------------------------------

# Extra feature: efficiency per block
df['ygpb_per_frame'] = df['total_ygpb'] / df['blocking_frames']
FEAT_4 = 'ygpb_per_frame'

for col in [BES_COL, FEAT_1, FEAT_2, FEAT_3, FEAT_4]:
    assert col in df.columns, f"Column '{col}' not found."

# ──────────────────────────────────────────────────────────
# 2. BINARY TARGET  (above-median = 1)
# ──────────────────────────────────────────────────────────
median_bes = df[BES_COL].median()
df['y'] = (df[BES_COL] > median_bes).astype(int)

print(f"Median {BES_COL} = {median_bes:.4f}\n")
print(df[[BES_COL, FEAT_1, FEAT_2, FEAT_3, FEAT_4, 'y']].head(), '\n')

# ──────────────────────────────────────────────────────────
# 3. SPLIT  +  MODEL
# ──────────────────────────────────────────────────────────
X = df[[FEAT_1, FEAT_2, FEAT_3, FEAT_4]]
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

clf = DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=42
     )
clf.fit(X_train, y_train)

# ──────────────────────────────────────────────────────────
# 4. EVALUATE
# ──────────────────────────────────────────────────────────
y_pred = clf.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred), '\n')
print("Classification Report:\n", classification_report(y_test, y_pred))

print("\nDECISION-TREE RULES\n")
print(export_text(clf, feature_names=[FEAT_1, FEAT_2, FEAT_3, FEAT_4]))
plt.figure(figsize=(12, 8))
plot_tree(
    clf,
    feature_names=[FEAT_1, FEAT_2, FEAT_3, FEAT_4],
    class_names=['Below Median', 'Above Median'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree for Volume-Adjusted BES Classification", fontsize=14)
plt.tight_layout

plt.savefig("/Users/derek/Downloads/bes_tree.png", dpi=300)
plt.close()

Median Volume_Adjusted_Skill_Player_Downfield_BES = 0.8324

   Volume_Adjusted_Skill_Player_Downfield_BES  median_ygpb  \
0                                    0.908735        4.245   
1                                    0.907261        2.120   
2                                    0.904727        1.760   
3                                    0.902747        2.080   
4                                    0.901406        2.350   

   normalized_block_volume  median_Skill_Player_Downfield_BES  ygpb_per_frame  \
0                 0.329105                           0.874276        4.245000   
1                 0.935457                           0.802858        2.017812   
2                 0.972035                           0.801098        2.156261   
3                 0.877332                           0.803685        2.305000   
4                 0.224649                           0.881601        2.350000   

   y  
0  1  
1  1  
2  1  
3  1  
4  1   

Accuracy: 84.13%

Confusion Matrix:
